# Part 1: Video Platform Interaction Behaviour

---

### Install Python packages (pip only)

In [2]:
%pip install networkx matplotlib numpy scipy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Import Python packages

In [3]:
import networkx as nx
import scipy.stats as stats

---

##### Examine the Graph Modelling Language (gml) files "social_network.gml" (social network) and "comment_network.gml" (comment network) which represent data for a sample of users on an online video sharing platform. Both networks are directed and share the same ids for nodes (anonymised users). However, the shared user ids are contained within the "label" attribute in the .gml files, not the node "id" attribute. Assume that all users have posted at least one video on the platform.

##### In the social network, an edge from a node, 𝑢, to some other node, 𝑣, indicates that 𝑢 subscribes to 𝑣 and their video content on the platform.

##### In the comment network, an edge from a node, 𝑢, to some other node, 𝑣, indicates that 𝑢 posted a comment to one or more videos posted made by 𝑣. Edges are weighted with the weight representing the number of times this happened over the time period the dataset represents.

##### Using these networks, answer the following questions:

##### Q1. How does the topological structure of the comment network differ from the social network in terms of the overall sparsity of edges between users and the number of connected groups of users?

In [9]:
# Load both networks from the gml
social_G = nx.read_gml("social_network.gml",label="label")
comment_G = nx.read_gml("comment_network.gml",label="label")
# Load both networks from the gml

# Helper: 2dp if >= 0.01, else scientific notation
def fmt(v):
        return f"{v:.2f}" if v >= 0.01 else f"{v:.2e}"

# --- Density ---
social_density = nx.density(social_G)
comment_density = nx.density(comment_G)

print("Social Network:")
print(f"Nodes: {social_G.number_of_nodes()}")
print(f"Edges: {social_G.number_of_edges()}")
print(f"Density: {fmt(social_density)}")
print()

print("Comment Network:")
print(f"Nodes: {comment_G.number_of_nodes()}")
print(f"Edges: {comment_G.number_of_edges()}")
print(f"Density: {fmt(comment_density)}")
print()

# Connected components
social_wcc = nx.number_weakly_connected_components(social_G)
comment_wcc = nx.number_weakly_connected_components(comment_G)

social_giant = max(len(c) for c in nx.weakly_connected_components(social_G))
comment_giant = max(len(c) for c in nx.weakly_connected_components(comment_G))

print("Weakly Connected Components")
print(f"Social network: number of weakly connected components: {social_wcc}")
print(f"Social network: largest component size: {social_giant}")
print(f"Comment network: number of weakly connected components: {comment_wcc}")
print(f"Comment network: largest component size: {comment_giant}")


Social Network:
Nodes: 25154
Edges: 311800
Density: 4.93e-04

Comment Network:
Nodes: 25154
Edges: 14425
Density: 2.28e-05

Weakly Connected Components
Social network: number of weakly connected components: 108
Social network: largest component size: 24934
Comment network: number of weakly connected components: 12427
Comment network: largest component size: 4509


##### Q2. Do users typically only comment on other users' videos yet do not have any on their own videos, only receive comments by others on their videos but do not comment on others, both comment and receive comments, or neither comment nor receive comments?

In [4]:
nodeOrder = list(comment_G.nodes())

only_commenter = []       # out > 0, in = 0: only comments on others
only_receives = []        # in > 0, out = 0: only receives comments
both = []                 # in > 0, out > 0: both comment and receive
neither = []              # in = 0, out = 0: neither

for node in nodeOrder:
    in_degree = comment_G.in_degree(node)
    out_degree = comment_G.out_degree(node)

    if in_degree > 0 and out_degree == 0:
        only_receives.append(node)

    elif in_degree == 0 and out_degree > 0:
        only_commenter.append(node)

    elif in_degree > 0 and out_degree > 0:
        both.append(node)
        
    else:
        neither.append(node)

total = len(nodeOrder)
print(f"Total users: {total}")
print(f"Only comment on others (out only): {len(only_commenter)} ({100*len(only_commenter)/total:.2f}%)")
print(f"Only receive comments  (in only):  {len(only_receives)} ({100*len(only_receives)/total:.2f}%)")
print(f"Both comment and receive:          {len(both)} ({100*len(both)/total:.2f}%)")
print(f"Neither comment nor receive:       {len(neither)} ({100*len(neither)/total:.2f}%)")


Total users: 25154
Only comment on others (out only): 9424 (37.47%)
Only receive comments  (in only):  5887 (23.40%)
Both comment and receive:          3139 (12.48%)
Neither comment nor receive:       6704 (26.65%)


##### Q3. How many users only comment on videos where they also subscribe to the user who posted the video?

In [5]:
count = 0
for user in comment_G.nodes():
    comment_targets = set(comment_G.successors(user))
    if len(comment_targets) == 0:
        continue
    subscribed_to = set(social_G.successors(user))
    if comment_targets.issubset(subscribed_to):
        count += 1

print(f"Users who only comment on videos by users they subscribe to: {count}")



Users who only comment on videos by users they subscribe to: 7914


##### Q4. How many users have only mutual subscription connections (i.e., every user that they subscribe to also subscribes to them) and only mutual comment connections with these same users?

In [12]:
mutual_count = 0

for u in social_G.nodes():
    
    social_out = set(social_G.successors(u))
    social_in  = set(social_G.predecessors(u))

    if len(social_out) == 0:
        continue
    if social_out != social_in:  # all subscriptions must be mutual
        continue

    comment_out = set(comment_G.successors(u))
    comment_in  = set(comment_G.predecessors(u))

    if comment_out != comment_in:  # all comment connections must be mutual
        continue
    if not comment_out.issubset(social_out):  # comment connections must be with same users
        continue


    mutual_count += 1

print(f"Users with only mutual subscriptions and mutual comments with those same users: {mutual_count}")

Users with only mutual subscriptions and mutual comments with those same users: 141


##### Q5. Do the 25 users that subscribe to the most other users also comment the most on videos by other users?

In [7]:
social_out_deg  = dict(social_G.out_degree())
comment_out_deg = dict(comment_G.out_degree())

top25_subscribers = set(sorted(social_out_deg, key=social_out_deg.get, reverse=True)[:25])
top25_commenters  = set(sorted(comment_out_deg, key=comment_out_deg.get, reverse=True)[:25])

overlap = top25_subscribers & top25_commenters
print(f"Users in both top 25 subscribers and top 25 commenters: {len(overlap)}")


Users in both top 25 subscribers and top 25 commenters: 0


##### Q6. To what extent does the number of subscribers a user has in the social network correlate with the number of users that have commented on their videos?

In [8]:
nodeOrder = list(social_G.nodes())

social_in_deg  = [social_G.in_degree(n)  for n in nodeOrder]
comment_in_deg = [comment_G.in_degree(n) for n in nodeOrder]

r, p = stats.pearsonr(social_in_deg, comment_in_deg)
print(f"Pearson r: {r:.2f}, p={p:.4f}")

Pearson r: 0.52, p=0.0000
